In [ ]:
import pybamm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import dfols
import signal
from scipy.integrate import solve_ivp
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy import interpolate, integrate
from stopit import threading_timeoutable as timeoutable
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from batfuns import *
plt.rcParams = set_rc_params(plt.rcParams)
import winsound
from pybamm import exp, constants, Parameter
import pickle
eSOH_DIR = "../data/esoh_R/"
oCV_DIR = "../data/ocv/"
cyc_DIR = "../data/cycling/"
fig_DIR = "../figures/figures_paper/"
res_DIR = "../data/results_paper/"
resistance_DIR = "../data/resistance/"
%matplotlib widget

In [ ]:
parameter_values = get_parameter_values()

spm = pybamm.lithium_ion.SPM(
    {
        "SEI": "ec reaction limited",
        "loss of active material": "stress-driven",
        "lithium plating": "irreversible",
        "stress-induced diffusion": "false",
    }
)
# spm.print_parameter_info()
param=spm.param

In [ ]:
cell = 3
# sim_des = sim_des+'_cv'
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
Ns = np.insert(N[1:]-1,0,0)
eps_n_data,eps_p_data,c_rate_c,c_rate_d,dis_set,Temp,SOC_0 = init_exp(cell_no,dfe_0,spm,parameter_values)
sno = 15
sim_des = f'cond{sno}'
pybamm.set_logging_level("WARNING")
# pybamm.set_logging_level("NOTICE")
par_val = {}
par_val[15] = [4.0312e-08,1.8157e-07,1.0776,2.3586e-09,-4.9170e-09,-1.4406e-09]

parameter_values = get_parameter_values()
parameter_values.update(
    {
        "Negative electrode active material volume fraction": eps_n_data,
        "Positive electrode active material volume fraction": eps_p_data,
        "Initial temperature [K]": 273.15+25,
        "Ambient temperature [K]": 273.15+25,
        "Positive electrode LAM constant proportional term [s-1]": par_val[sno][0],
        "Negative electrode LAM constant proportional term [s-1]": par_val[sno][1],
        "Positive electrode LAM constant proportional term 2 [s-1]": par_val[sno][5],
        "Negative electrode LAM constant proportional term 2 [s-1]": par_val[sno][4],
        "Positive electrode LAM constant exponential term": par_val[sno][2],
        "Negative electrode LAM constant exponential term": par_val[sno][2],
        "SEI kinetic rate constant [m.s-1]":  4.60788219e-16, #1.08494281e-16 , 
        "EC diffusivity [m2.s-1]": 4.56607447e-19,#8.30909086e-19,
        "SEI growth activation energy [J.mol-1]": 1.87422275e+04,#1.58777981e+04,
        "Lithium plating kinetic rate constant [m.s-1]": par_val[sno][3],
        "Initial inner SEI thickness [m]": 0e-09,
        "Initial outer SEI thickness [m]": 5e-09,
        "SEI resistivity [Ohm.m]": 30000.0,
        "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
        "Negative electrode LAM min stress [Pa]": 0,
        "Negative electrode LAM max stress [Pa]": 0,
        "Positive electrode LAM min stress [Pa]": 0,
        "Positive electrode LAM max stress [Pa]": 0,
        "Negative electrode diffusion coefficient [m2.s-1]": 8e-14,
        "Positive electrode diffusion coefficient [m2.s-1]": 8e-15,
        # "Negative electrode critical stress [Pa]": 20e+06,
        # "Positive electrode critical stress [Pa]": 40e+06,
    },
    check_already_exists=False,
)
if cell == 13 or cell == 16:
    parameter_values.update(
        {
            "Negative electrode partial molar volume [m3.mol-1]":	0.747*7e-06,
        },
        check_already_exists=False,
    )

In [ ]:
experiment_rev_exp = pybamm.Experiment(
    [
        "Discharge at "+'C/20'+" until 3V",
        "Rest for 60 min",
        "Charge at "+'C/20'+" until 4.2V", 
        # "Hold at 4.2V until C/50"
    ],
    period = '10 s',
)

In [ ]:
def rev_exp_2(cyc_no,eSOH,eSOH_data,parameter_values):
    # dfo = dfo_0[dfo_0['N']==N[cyc_no]]
    model = spm
    Vmin = 3.0
    Vmax = 4.2
    esoh_model = pybamm.lithium_ion.ElectrodeSOH()
    esoh_sim = pybamm.Simulation(esoh_model, parameter_values=parameter_values)
    param = model.param
    Cn = eSOH["C_n"][cyc_no]
    # print(Cn)
    Cp = eSOH["C_p"][cyc_no]
    c_n_max = parameter_values.evaluate(param.n.prim.c_max)
    c_p_max = parameter_values.evaluate(param.p.prim.c_max)
    n_Li_init = eSOH["n_Li"][cyc_no]
    eps_n_data = parameter_values.evaluate(Cn*3600/(param.n.L * param.n.prim.c_max * param.F* param.A_cc))
    eps_p_data = parameter_values.evaluate(Cp*3600/(param.p.L * param.p.prim.c_max * param.F* param.A_cc))

    esoh_sol = esoh_sim.solve(
        [0],
        inputs={"V_min": Vmin, "V_max": Vmax, "C_n": Cn, "C_p": Cp, "n_Li": n_Li_init},
        solver=pybamm.AlgebraicSolver(),
    )

    parameter_values.update(
        {
            "Initial concentration in negative electrode [mol.m-3]": esoh_sol[
       "x_100"
            ].data[0]
            * c_n_max,
            "Initial concentration in positive electrode [mol.m-3]": esoh_sol[
                "y_100"
            ].data[0]
            * c_p_max,
            "Negative electrode active material volume fraction": eps_n_data,
            "Positive electrode active material volume fraction": eps_p_data,
            "Initial temperature [K]": 273.15+25,
            "Ambient temperature [K]": 273.15+25,
            
        }
    )

    sim_exp = pybamm.Simulation(
        model, experiment=experiment_rev_exp, parameter_values=parameter_values,
        solver=pybamm.CasadiSolver("safe")
    )
    sol_exp = sim_exp.solve()
    t_t = sol_exp["Time [s]"].entries
    I_t = sol_exp["Current [A]"].entries
    Q_t = -sol_exp['Discharge capacity [A.h]'].entries
    Vt_t = sol_exp["Terminal voltage [V]"].entries
    exp_t = 30e6*sol_exp["Cell thickness change [m]"].entries
    idx = np.where(np.diff(np.sign(I_t)))[0]
    I = I_t[idx[-1]:]
    t = t_t[idx[-1]:]-t_t[idx[-1]]
    Q = Q_t[idx[-1]:]-Q_t[idx[-1]]
    Vt = Vt_t[idx[-1]:]
    Exp = exp_t[idx[-1]:]-exp_t[idx[-1]]
    # Q_filt = ((Q/max(Q))>0.15)
    # int_V = interpolate.CubicSpline(eSOH_data['Q'],eSOH_data['V'],extrapolate=True)
    # rmse_V = pybamm.rmse(Vt*Q_filt,int_V(Q)*Q_filt)
    # V_err = Vt*Q_filt-int_V(Q)*Q_filt
    # V_max_loc = np.argmax(abs(V_err))
    # V_err_Q = Q[V_max_loc]
    # V_err_max = V_err[V_max_loc]
    if max(Q)<max(eSOH_data['Q']):
        int_V = interpolate.CubicSpline(eSOH_data['Q'],eSOH_data['V'],extrapolate=True)
        rmse_V = pybamm.rmse(Vt,int_V(Q))
        V_err = Vt-int_V(Q)
        max_V = abs(V_err[-2])
        V_max_loc = np.argmax(abs(V_err))
        V_err_Q = Q[V_max_loc]
        V_err_max = V_err[V_max_loc]
        int_E = interpolate.CubicSpline(eSOH_data['Q'],eSOH_data['E']-eSOH_data['E'].iloc[0],extrapolate=True)
        rmse_E = pybamm.rmse(Exp,int_E(Q))
        E_err = Exp-int_E(Q)
        max_E = abs(E_err[-2])
        E_max_loc = np.argmax(abs(E_err))
        E_err_Q = Q[E_max_loc]
        E_err_max = E_err[E_max_loc]
        Q_err = Q
    else:
        int_V = interpolate.CubicSpline(Q,Vt,extrapolate=True)
        rmse_V = pybamm.rmse(eSOH_data['V'],int_V(eSOH_data['Q']))
        V_err = int_V(eSOH_data['Q'])-eSOH_data['V']
        max_V = abs(V_err.iloc[-2])
        V_max_loc = np.argmax(abs(V_err))
        V_err_Q = eSOH_data['Q'].iloc[V_max_loc]
        V_err_max = V_err.iloc[V_max_loc]
        int_E = interpolate.CubicSpline(Q,Exp,extrapolate=True)
        rmse_E = pybamm.rmse(eSOH_data['E']-eSOH_data['E'].iloc[0],int_E(eSOH_data['Q']))
        E_err = int_E(eSOH_data['Q'])-eSOH_data['E']+eSOH_data['E'].iloc[0]
        max_E = abs(E_err.iloc[-2])
        E_max_loc = np.argmax(abs(E_err))
        E_err_Q = eSOH_data['Q'].iloc[E_max_loc]
        E_err_max = E_err.iloc[E_max_loc]
        Q_err = eSOH_data['Q']
    return t,I,Q,Vt,Exp,sol_exp,rmse_V,rmse_E

In [ ]:
cells = [3,6,9,12,15,18]
# cells = [2,5,8,11,14,17]
for cell in cells:
    # sim_des = sim_des+'_cv'
    cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
    print(cell_no)
    Ns = np.insert(N[1:]-1,0,0)
    eps_n_data,eps_p_data,c_rate_c,c_rate_d,dis_set,Temp,SOC_0 = init_exp(cell_no,dfe_0,spm,parameter_values)
    i = 0
    titles = ["BOL","MOL","EOL"]
    fig, axes = plt.subplots(2,3,figsize=(10,6))
    for cyc_no in [0,int((len(N)-1)/2),len(N)-1]:
    # for cyc_no in range(0,int((len(N)+1)/2),len(N)-1):
        print(cyc_no)
        # dfo = dfo_0[dfo_0['N']==N_0[cyc_no+1]]
        dfo = dfo_0[dfo_0['N']==N_0[cyc_no]]
        parameter_values.update(
        {
            "Positive electrode LAM constant proportional term [s-1]": par_val[15][0],
            "Negative electrode LAM constant proportional term [s-1]": par_val[15][1],
            "Positive electrode LAM constant exponential term": par_val[15][2],
            "Negative electrode LAM constant exponential term": par_val[15][2],
            "Lithium plating kinetic rate constant [m.s-1]": par_val[15][3],
        },
        check_already_exists=False,
        )
        # t,I,Q,Vt,Exp,sol_exp,rmse_V,rmse_E = rev_exp(cyc_no,all_sumvars_dict,dfo,parameter_values)
        t,I,Q,Vt,Exp,sol_exp,rmse_V,rmse_E = rev_exp_2(cyc_no,dfe_0,dfo,parameter_values)
        ax1 = axes.flat[i+3]
        df = pd.DataFrame({'t': t,'I': I, 'Q': Q,'Vt':Vt,'Exp':Exp,
                    })
        df.to_csv(res_DIR + "volt_exp_sim_c20_cell_"+cell_no+"_cyc_"+f"{N[cyc_no]}"+".csv", index=False)
        ax1.plot(dfo['Q'],dfo['E']-dfo['E'].iloc[0],'k',linewidth=1)
        ax1.plot(Q,Exp,'r--',linewidth=1.5)
        ax1.set_ylim([0,110])
        ax1.set_xlim([0,5.1])
        ax1.set_xticklabels([])
        
        ax1.text(0.1,0.9,f'RMSE: {rmse_E:0.1f} $\mu$m',transform=ax1.transAxes)
        ax2 = axes.flat[i]
        ax2.plot(dfo['Q'],dfo['V'],'k',linewidth=1)
        ax2.plot(Q,Vt,'r--',linewidth=1.5)
        ax2.set_ylim([3,4.2])
        ax2.set_xlim([0,5.1])
        ax1.set_xlabel('Q [Ah]')
        ax2.text(0.5,0.1,f'RMSE: {rmse_V*1e3:0.1f} mV',transform=ax2.transAxes)
        ax2.set_title('{} [N:{:0.0f},Ah:{:0.0f},%Cap:{:0.1f}]'.format(titles[i],N[cyc_no],dfe['Ah_th'][cyc_no],(dfe["Cap"][cyc_no]/dfe["Cap"][0])*100))
        if i ==0:
            ax2.set_ylabel('$V_t$ [Ah]')
            ax1.set_ylabel('$\Delta$t [$\mu$m]')
        i+=1
    ax2.legend(["Data","Sim"])
    # fig.suptitle('C/20 Charge Voltage & Reversible Expansion')
    fig.tight_layout()
    plt.savefig(fig_DIR +'c20_volt_exp_cell_'+cell_no+'.png')

In [ ]:
titles = ["BOL","MOL","EOL"]
linestyles = ["solid","dashed","dotted"]
sno = 15
sim_des = f'cond{sno}'
i = 0
fig, ax = plt.subplots(2,3,figsize=(12,6))
cell = 2
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
Ns = np.insert(N[1:]-1,0,0)
cycles = [0,int((len(N)-1)/2),len(N)-1]
cyc_no = cycles[0]
ax2 = ax.flat[0]
dfo = dfo_0[dfo_0['N']==N_0[cyc_no]]
Q_d = dfo['Q']
V_d = dfo['V']
E_d = dfo['E']-dfo['E'].iloc[0]
df = pd.read_csv(res_DIR + "volt_exp_sim_c20_cell_"+cell_no+"_cyc_"+f"{N[cyc_no]}"+".csv")
ax2.plot(Q_d,V_d,'k',linewidth=2)
ax2.plot(df['Q'],df["Vt"],'r',linewidth=2,linestyle=linestyles[0])
ax2.set_xlabel('Q [Ah]')
# ax2.set_ylim(bottom=0)
ax2.set_ylabel('$V_t$ [V]')
cyc_no = cycles[1]
dfo = dfo_0[dfo_0['N']==N_0[cyc_no]]
Q_d = dfo['Q']
V_d = dfo['V']
E_d = dfo['E']-dfo['E'].iloc[0]
df = pd.read_csv(res_DIR + "volt_exp_sim_c20_cell_"+cell_no+"_cyc_"+f"{N[cyc_no]}"+".csv")
ax2.plot(Q_d,V_d,'k',linewidth=2,label="_nolegend_")
ax2.plot(df['Q'],df["Vt"],'r',linewidth=2,linestyle=linestyles[1])
cyc_no = cycles[2]
dfo = dfo_0[dfo_0['N']==N_0[cyc_no]]
Q_d = dfo['Q']
V_d = dfo['V']
E_d = dfo['E']-dfo['E'].iloc[0]
df = pd.read_csv(res_DIR + "volt_exp_sim_c20_cell_"+cell_no+"_cyc_"+f"{N[cyc_no]}"+".csv")
ax2.plot(Q_d,V_d,'k',linewidth=2,label="_nolegend_")
ax2.plot(df['Q'],df["Vt"],'r',linewidth=2,linestyle=linestyles[2])
ax2.legend(["Data","BOL","MOL","EOL"])
cells = [2,1,3]
for cell in cells:
    j=0
    cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
    cycles = [0,int((len(N)-1)/2),len(N)-1]
    if cell == 1:
        cycles = [0,int((len(N)+1)/2),len(N)-2]
    for cyc_no in cycles:
        print(cyc_no)
        df = pd.read_csv(res_DIR + "volt_exp_sim_c20_cell_"+cell_no+"_cyc_"+f"{N[cyc_no]}"+".csv")
        dfo = dfo_0[dfo_0['N']==N_0[cyc_no]]
        if cell == 1:
            dfo = dfo_0[dfo_0['N']==N_0[cyc_no+1]]
        Q_d = dfo['Q']
        V_d = dfo['V']
        E_d = dfo['E']-dfo['E'].iloc[0]
        if cell == 2:
            pass
        else:
            ax1 = ax.flat[i]
            ax1.plot(Q_d,V_d,'k',linewidth=2)
            ax1.plot(df['Q'],df["Vt"],'r',linewidth=2,linestyle=linestyles[j])
            ax1.set_xlabel('Q [Ah]')
            # ax1.text(0.2,0.38-0.13*j,'{}- N:{:03.0f},Ah:{:04.0f},%Cap:{:03.0f}'.format(titles[j],N[cyc_no],dfe['Ah_th'][cyc_no],(dfe["Cap"][cyc_no]/dfe["Cap"][0])*100),transform=ax1.transAxes,family='monospace',fontsize='10')
            ax1.set_ylabel('$V_t$ [V]')
        ax3 = ax.flat[i+3]
        ax3.plot(Q_d,E_d,'k',linewidth=2)
        ax3.plot(df['Q'],df["Exp"],'r',linewidth=2,linestyle=linestyles[j])
        ax3.set_xlabel('Q [Ah]')
        # ax3.set_ylim([0,275])
        ax3.set_ylabel('$\delta$ [$\mu$m]')
        # ax3.text(0.5,0.2,f'RMSE: {rmse_E:0.1f}'+ r'$\mu$m',transform=ax3.transAxes)
        j+=1
    i+=1
ax10 = ax.flat[0]
# ax10.text(-0.2,1.2,f'a) Current',transform=ax10.transAxes,fontsize=16)
ax10.set_title(r"Cold (-5$^\circ$C)")
ax11 = ax.flat[1]
ax11.set_title(r"Room (25$^\circ$C)")
ax12 = ax.flat[2]
ax12.set_title(r"Hot (45$^\circ$C)")
# ax20 = ax.flat[3]
# ax20.text(-0.2,1.1,f'b) Voltage',transform=ax20.transAxes,fontsize=16)
# ax30 = ax.flat[6]
# ax30.text(-0.2,1.1,f'c) Total Expansion',transform=ax30.transAxes,fontsize=16)
# ax3.legend(["Data","Sim"])
fig.tight_layout()
plt.savefig(fig_DIR +'volt_exp_c20_error_c5.png')

In [ ]:
titles = ["BOL","MOL","EOL"]
linestyles = ["solid","dashed","dotted"]
sno = 15
sim_des = f'cond{sno}'
i = 0
fig, ax = plt.subplots(2,3,figsize=(12,6))
cell = 2+3
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
Ns = np.insert(N[1:]-1,0,0)
cycles = [0,int((len(N)-1)/2),len(N)-1]
cyc_no = cycles[0]
ax2 = ax.flat[0]
dfo = dfo_0[dfo_0['N']==N_0[cyc_no]]
Q_d = dfo['Q']
V_d = dfo['V']
E_d = dfo['E']-dfo['E'].iloc[0]
df = pd.read_csv(res_DIR + "volt_exp_sim_c20_cell_"+cell_no+"_cyc_"+f"{N[cyc_no]}"+".csv")
ax2.plot(Q_d,V_d,'k',linewidth=2)
ax2.plot(df['Q'],df["Vt"],'r',linewidth=2,linestyle=linestyles[0])
ax2.set_xlabel('Q [Ah]')
# ax2.set_ylim(bottom=0)
ax2.set_ylabel('$V_t$ [V]')
cyc_no = cycles[1]
dfo = dfo_0[dfo_0['N']==N_0[cyc_no]]
Q_d = dfo['Q']
V_d = dfo['V']
E_d = dfo['E']-dfo['E'].iloc[0]
df = pd.read_csv(res_DIR + "volt_exp_sim_c20_cell_"+cell_no+"_cyc_"+f"{N[cyc_no]}"+".csv")
ax2.plot(Q_d,V_d,'k',linewidth=2,label="_nolegend_")
ax2.plot(df['Q'],df["Vt"],'r',linewidth=2,linestyle=linestyles[1])
cyc_no = cycles[2]
dfo = dfo_0[dfo_0['N']==N_0[cyc_no]]
Q_d = dfo['Q']
V_d = dfo['V']
E_d = dfo['E']-dfo['E'].iloc[0]
df = pd.read_csv(res_DIR + "volt_exp_sim_c20_cell_"+cell_no+"_cyc_"+f"{N[cyc_no]}"+".csv")
ax2.plot(Q_d,V_d,'k',linewidth=2,label="_nolegend_")
ax2.plot(df['Q'],df["Vt"],'r',linewidth=2,linestyle=linestyles[2])
ax2.legend(["Data","BOL","MOL","EOL"])
cells = [5,4,6]
for cell in cells:
    j=0
    cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
    cycles = [0,int((len(N)-1)/2),len(N)-1]
    if cell == 4:
        cycles = [0,int((len(N)+1)/2),len(N)-1]
    for cyc_no in cycles:
        print(cyc_no)
        df = pd.read_csv(res_DIR + "volt_exp_sim_c20_cell_"+cell_no+"_cyc_"+f"{N[cyc_no]}"+".csv")
        dfo = dfo_0[dfo_0['N']==N_0[cyc_no]]
        if cell == 4:
            dfo = dfo_0[dfo_0['N']==N_0[cyc_no+1]]
        Q_d = dfo['Q']
        V_d = dfo['V']
        E_d = dfo['E']-dfo['E'].iloc[0]
        if cell == 2:
            pass
        else:
            ax1 = ax.flat[i]
            ax1.plot(Q_d,V_d,'k',linewidth=2)
            ax1.plot(df['Q'],df["Vt"],'r',linewidth=2,linestyle=linestyles[j])
            ax1.set_xlabel('Q [Ah]')
            # ax1.text(0.2,0.38-0.13*j,'{}- N:{:03.0f},Ah:{:04.0f},%Cap:{:03.0f}'.format(titles[j],N[cyc_no],dfe['Ah_th'][cyc_no],(dfe["Cap"][cyc_no]/dfe["Cap"][0])*100),transform=ax1.transAxes,family='monospace',fontsize='10')
            ax1.set_ylabel('$V_t$ [V]')
        ax3 = ax.flat[i+3]
        ax3.plot(Q_d,E_d,'k',linewidth=2)
        ax3.plot(df['Q'],df["Exp"],'r',linewidth=2,linestyle=linestyles[j])
        ax3.set_xlabel('Q [Ah]')
        # ax3.set_ylim([0,275])
        ax3.set_ylabel('$\delta$ [$\mu$m]')
        # ax3.text(0.5,0.2,f'RMSE: {rmse_E:0.1f}'+ r'$\mu$m',transform=ax3.transAxes)
        j+=1
    i+=1
ax10 = ax.flat[0]
# ax10.text(-0.2,1.2,f'a) Current',transform=ax10.transAxes,fontsize=16)
ax10.set_title(r"Cold (-5$^\circ$C)")
ax11 = ax.flat[1]
ax11.set_title(r"Room (25$^\circ$C)")
ax12 = ax.flat[2]
ax12.set_title(r"Hot (45$^\circ$C)")
# ax20 = ax.flat[3]
# ax20.text(-0.2,1.1,f'b) Voltage',transform=ax20.transAxes,fontsize=16)
# ax30 = ax.flat[6]
# ax30.text(-0.2,1.1,f'c) Total Expansion',transform=ax30.transAxes,fontsize=16)
# ax3.legend(["Data","Sim"])
fig.tight_layout()
plt.savefig(fig_DIR +'volt_exp_c20_error_1p5C.png')